> Legacy/optional workflow. For author SLM-SQL + CSC baseline and CSC LoRA research, use **09 → 10 → 11** and `docs/CSC_LORA.md`. This notebook is not the canonical comparison against the author system.

# 05 — Adaptive Sampling & Pareto Analysis

Replay policies on one pre-generated candidate artifact, run ablations, and build Pareto curves. CSC training is opt-in; the router simulation reports CSC decision points, not a live merge-model call.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path


def bootstrap_repo():
    configured = os.environ.get('SLM_PRJ_PATH')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents,
                       Path('/kaggle/working/SLM-PRJ'),
                       Path('/kaggle/working/SLMSQL_project')])
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    target = Path('/kaggle/working/SLM-PRJ')
    url = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')
    subprocess.run(['git', 'clone', url, str(target)], check=True)
    return target

REPO_DIR = bootstrap_repo()
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import, resolve_data_dir
REPO_DIR = prepare_repo_import(REPO_DIR)
DATA_DIR = resolve_data_dir(REPO_DIR)

BIRD_DIR = DATA_DIR / 'bird'
print(f'Repo: {REPO_DIR}')

## 1. Offline Policy Simulation

Replay stopping policies on one pre-generated candidate artifact (N=32 per question, 100+ permutations). This is deliberate artifact reuse, not a second model-generation pass.


In [ ]:
from benchmarks.bird_loader import get_db_path
from research.oracle_analysis import load_exec_results

EXEC_FILE = Path(os.environ.get('EXEC_FILE', str(REPO_DIR / 'results' / 'candidates' / 'exec_dev.jsonl')))
RESULTS_DIR = REPO_DIR / 'results' / 'policies'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not EXEC_FILE.is_file():
    raise FileNotFoundError(f'Execution results not found: {EXEC_FILE}. Run notebook 04 first.')

records = load_exec_results(EXEC_FILE)
db_path_fn = lambda db_id: get_db_path(BIRD_DIR, db_id)
print(f'Loaded {len(records)} questions with candidates')


In [ ]:
from research.policies.fixed_k import FixedKPolicy
from research.policies.beta_stopping import BetaStoppingPolicy
from research.policies.margin_entropy import MarginEntropyPolicy
from research.simulator import simulate_multiple_policies

n_permutations = int(os.environ.get('N_PERMUTATIONS', '100'))
policies = {
    'fixed_k4': FixedKPolicy(k=4),
    'fixed_k8': FixedKPolicy(k=8),
    'fixed_k16': FixedKPolicy(k=16),
    'fixed_k32': FixedKPolicy(k=32),
    'beta_095': BetaStoppingPolicy(threshold=0.95, min_k=4, max_k=32),
    'beta_090': BetaStoppingPolicy(threshold=0.90, min_k=4, max_k=32),
    'margin_3': MarginEntropyPolicy(margin=3, entropy_threshold=1.0, min_k=4, max_k=32),
    'margin_5': MarginEntropyPolicy(margin=5, entropy_threshold=0.8, min_k=4, max_k=32),
}

sim_results = simulate_multiple_policies(
    records, policies, db_path_fn, n_permutations=n_permutations,
)

print(f'{"Policy":<15} {"EX":>8} {"CI95":>10} {"Avg_K":>8}')
print('-' * 43)
for name, result in sim_results.items():
    print(f'{name:<15} {result["ex_mean"]:>8.4f} ±{result["ex_ci95"]:.4f} {result["avg_samples"]:>8.1f}')


In [ ]:
with open(RESULTS_DIR / 'simulation_results.json', 'w') as f:
    json.dump(sim_results, f, indent=2)
print('Saved simulation results')

## 2. Learned Stopper (Logistic + XGBoost)

In [ ]:
from research.policies.learned_stopper import LearnedStopperPolicy
from research.calibration import compute_calibration_data

calib_data = compute_calibration_data(records, db_path_fn)
for model_type in ['logistic', 'xgboost']:
    policy = LearnedStopperPolicy(model_type=model_type, min_k=4, max_k=32)
    policy.train(calib_data)
    result = simulate_multiple_policies(
        records, {f'learned_{model_type}': policy}, db_path_fn,
        n_permutations=int(os.environ.get('N_PERMUTATIONS', '100')),
    )[f'learned_{model_type}']
    print(f'{model_type:10s} EX={result["ex_mean"]:.4f} Avg_K={result["avg_samples"]:.1f}')
    sim_results[f'learned_{model_type}'] = result

with (RESULTS_DIR / 'simulation_results.json').open('w', encoding='utf-8') as handle:
    json.dump(sim_results, handle, indent=2)


## 3. Candidate Quality Ablation

In [ ]:
from research.candidate_quality import run_ablation

ablation_results = run_ablation(records, db_path_fn)

print(f'{"Config":<30} {"EX":>8} {"N":>6}')
print('-' * 46)
for label, metrics in ablation_results.items():
    print(f'{label:<30} {metrics["ex"]:>8.4f} {metrics["n"]:>6}')

with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

## 4. CSC adaptation moved to notebooks 09–11

No training pairs are built from these dev artifacts. The canonical workflow adapts the author merger on database-disjoint train/validation partitions.


In [ ]:
print('Legacy CSC training from dev artifacts is disabled. Use notebooks 09–11 with training-only pairs.')


In [ ]:
print('Legacy CSC training from dev artifacts is disabled. Use notebooks 09–11 with training-only pairs.')


## 5. Three-Action Router Simulation

In [ ]:
from research.policies.three_action_router import ThreeActionRouterPolicy

router_configs = {
    'router_default': ThreeActionRouterPolicy(stop_threshold=0.90, csc_threshold=0.70, min_k=4, max_k=32),
    'router_aggressive': ThreeActionRouterPolicy(stop_threshold=0.85, csc_threshold=0.50, min_k=4, max_k=24),
    'router_conservative': ThreeActionRouterPolicy(stop_threshold=0.95, csc_threshold=0.80, min_k=8, max_k=32),
}
router_results = simulate_multiple_policies(
    records, router_configs, db_path_fn,
    n_permutations=int(os.environ.get('N_PERMUTATIONS', '100')),
)
print(f'{"Router":<25} {"EX":>8} {"Avg_K":>8} {"CSC_decisions":>14}')
print('-' * 60)
for name, result in router_results.items():
    print(f'{name:<25} {result["ex_mean"]:>8.4f} {result["avg_samples"]:>8.1f} {result.get("csc_calls", "N/A"):>14}')
    sim_results[name] = result

with (RESULTS_DIR / 'simulation_results.json').open('w', encoding='utf-8') as handle:
    json.dump(sim_results, handle, indent=2)

## 6. Pareto Analysis

Aggregate all experiments into EX vs cost Pareto curves.

In [ ]:
from research.pareto import build_pareto_table, pareto_frontier, save_pareto_report, plot_pareto

FINAL_DIR = REPO_DIR / 'results' / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
all_experiments = []
for name, result in sim_results.items():
    all_experiments.append({
        'name': name,
        'ex_mean': result.get('ex_mean', 0),
        'avg_samples': result.get('avg_samples', 0),
        'avg_tokens': result.get('avg_tokens', 0),
        'avg_seconds': result.get('avg_seconds', 0),
    })

rows = build_pareto_table(all_experiments)
save_pareto_report(rows, FINAL_DIR / 'pareto_report.json')
print('Pareto report saved')
frontier = pareto_frontier(rows)
for row in frontier:
    print(f'  {row["name"]}: EX={row["ex"]:.4f}, samples={row["avg_samples"]:.1f}')


In [ ]:
plot_pareto(rows, x_key='avg_samples', y_key='ex', output_path=FINAL_DIR / 'pareto_samples.png')
plot_pareto(rows, x_key='avg_tokens', y_key='ex', output_path=FINAL_DIR / 'pareto_tokens.png')
plot_pareto(rows, x_key='avg_seconds', y_key='ex', output_path=FINAL_DIR / 'pareto_seconds.png')
print(f'Pareto plots saved to {FINAL_DIR}')


## 7. Summary Table

In [ ]:
print(f'{"Experiment":<25} {"EX":>8} {"Avg_K":>8} {"CI95":>10}')
print('=' * 53)

sorted_results = sorted(sim_results.items(), key=lambda x: x[1].get('ex_mean', 0), reverse=True)
for name, res in sorted_results:
    print(f'{name:<25} {res.get("ex_mean",0):>8.4f} {res.get("avg_samples",0):>8.1f} '
          f'±{res.get("ex_ci95",0):>8.4f}')

best = sorted_results[0]
print(f'\nBest policy: {best[0]} with EX={best[1]["ex_mean"]:.4f} using {best[1]["avg_samples"]:.1f} avg samples')